In [ ]:
# Structure
# CELL 1: Install packages
# CELL 2: Import libraries
# CELL 3: All class definitions
# CELL 4: Your API keys
# CELL 5: Run trading (Run daily to rebalance)
# CELL 6: View performance
# CELL 7: Manual commands
# CELL 8; Risk adjusted performance metrics
# CELL 9: 5 year backtest with statistical significance

In [32]:
%pip install alpaca-py yfinance pandas numpy matplotlib
%pip install alpaca-trade-api --upgrade

print("All required packages are installed.")

Defaulting to user installation because normal site-packages is not writeable
  Using cached websockets-15.0.1-cp39-cp39-macosx_11_0_arm64.whl (173 kB)
  Attempting uninstall: websockets
    Found existing installation: websockets 10.4
    Uninstalling websockets-10.4:
      Successfully uninstalled websockets-10.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
alpaca-trade-api 3.2.0 requires websockets<11,>=9.0, but you have websockets 15.0.1 which is incompatible.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
  Using cached websockets-10.4-cp39-cp39-macosx_11_0_arm64.whl (97 kB)
  Attempting uninstall: websockets
    Found existing in

In [33]:
# install alpaca-trade-api yfinance pandas numpy matplotlib

try:
    import alpaca
except ImportError:
    !python3 -m pip install alpaca-py
    import alpaca
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json
from datetime import datetime, timedelta
import alpaca_trade_api as tradeapi

print("✓ All libraries imported successfully")


✓ All libraries imported successfully


In [35]:
# ALPACA PAPER TRADING ENGINE
# ============================================================================

class AlpacaPaperTrading:
    """Automated paper trading via Alpaca API"""
    
    def __init__(self, api_key=None, secret_key=None):
        """Initialize Alpaca paper trading client"""
        
        self.api_key = api_key or os.getenv('ALPACA_API_KEY')
        self.secret_key = secret_key or os.getenv('ALPACA_SECRET_KEY')
        
        if not self.api_key or not self.secret_key:
            raise ValueError("Alpaca API keys required. Pass them as: AlpacaPaperTrading(api_key='...', secret_key='...')")
        
        # Initialize API client
        self.api = tradeapi.REST(
            key_id=self.api_key,
            secret_key=self.secret_key,
            base_url='https://paper-api.alpaca.markets',  # Paper trading endpoint
            api_version='v2'
        )
        
        print("✓ Connected to Alpaca Paper Trading")
    
    def get_account(self):
        """Get account information"""
        account = self.api.get_account()
        
        print(f"\n{'='*70}")
        print("ALPACA PAPER TRADING ACCOUNT")
        print(f"{'='*70}")
        print(f"Buying Power: ${float(account.buying_power):,.2f}")
        print(f"Cash: ${float(account.cash):,.2f}")
        print(f"Portfolio Value: ${float(account.portfolio_value):,.2f}")
        print(f"Account Status: {account.status}")
        
        return account
    
    def get_positions(self):
        """Get current positions"""
        positions = self.api.list_positions()
        
        if not positions:
            print("\nNo current positions")
            return {}
        
        print(f"\n{'='*70}")
        print("CURRENT POSITIONS")
        print(f"{'='*70}")
        
        positions_dict = {}
        
        for position in positions:
            ticker = position.symbol
            shares = float(position.qty)
            current_price = float(position.current_price)
            market_value = float(position.market_value)
            unrealized_pl = float(position.unrealized_pl)
            unrealized_plpc = float(position.unrealized_plpc)
            
            positions_dict[ticker] = {
                'shares': shares,
                'price': current_price,
                'value': market_value,
                'unrealized_pl': unrealized_pl,
                'unrealized_plpc': unrealized_plpc
            }
            
            print(f"{ticker}: {shares} shares @ ${current_price:.2f} = ${market_value:,.2f} ({unrealized_plpc:+.2%})")
        
        return positions_dict
    
    def place_order(self, ticker, shares, side):
        """Place market order"""
        
        try:
            order = self.api.submit_order(
                symbol=ticker,
                qty=abs(shares),
                side=side.lower(),
                type='market',
                time_in_force='day'
            )
            print(f"✓ {side} order placed: {abs(shares)} shares of {ticker}")
            return order
        except Exception as e:
            print(f"⚠️  Order failed for {ticker}: {str(e)}")
            return None
    
    def close_all_positions(self):
        """Close all current positions"""
        print(f"\n{'='*70}")
        print("CLOSING ALL POSITIONS")
        print(f"{'='*70}")
        
        try:
            self.api.close_all_positions()
            print("✓ All positions closed")
        except Exception as e:
            print(f"⚠️  Error closing positions: {str(e)}")
    
    def rebalance_to_target(self, target_weights):
        """Rebalance portfolio to target allocation"""
        
        print(f"\n{'='*70}")
        print("REBALANCING TO TARGET ALLOCATION")
        print(f"{'='*70}")
        
        # Get account info
        account = self.api.get_account()
        total_value = float(account.portfolio_value)
        
        print(f"\nTotal Portfolio Value: ${total_value:,.2f}")
        
        # Get current positions
        current_positions = self.get_positions()
        
        # Get current prices
        tickers = list(set(list(target_weights.keys()) + list(current_positions.keys())))
        
        print(f"\nTarget Allocation:")
        for ticker, weight in sorted(target_weights.items(), key=lambda x: x[1], reverse=True):
            target_value = total_value * weight
            print(f"  {ticker}: {weight:.2%} (${target_value:,.2f})")
        
        # Get latest prices
        price_data = yf.download(tickers, period='1d', progress=False)
        
        current_prices = {}
        if len(tickers) == 1:
            current_prices[tickers[0]] = price_data['Close'].iloc[-1]
        else:
            for ticker in tickers:
                try:
                    current_prices[ticker] = price_data['Close'][ticker].iloc[-1]
                except:
                    current_prices[ticker] = 0
        
        # Calculate trades
        trades = []
        
        for ticker, target_weight in target_weights.items():
            target_value = total_value * target_weight
            current_shares = current_positions.get(ticker, {}).get('shares', 0)
            current_price = current_prices.get(ticker, 0)
            
            if current_price == 0:
                print(f"⚠️  Skipping {ticker}: No price data")
                continue
            
            target_shares = int(target_value / current_price)
            shares_diff = target_shares - current_shares
            
            if abs(shares_diff) > 0:
                side = 'buy' if shares_diff > 0 else 'sell'
                trades.append({
                    'ticker': ticker,
                    'shares': abs(shares_diff),
                    'side': side,
                    'priority': 0 if side == 'sell' else 1
                })
        
        # Close positions not in target
        for ticker in current_positions.keys():
            if ticker not in target_weights:
                shares = current_positions[ticker]['shares']
                trades.append({
                    'ticker': ticker,
                    'shares': shares,
                    'side': 'sell',
                    'priority': 0
                })
        
        # Execute trades (sells first)
        print(f"\nExecuting Trades:")
        for trade in sorted(trades, key=lambda x: x['priority']):
            self.place_order(
                ticker=trade['ticker'],
                shares=trade['shares'],
                side=trade['side']
            )
        
        print(f"\n✓ Rebalancing complete")
        
        # Wait for orders to fill
        import time
        time.sleep(3)
        
        self.get_positions()

# ============================================================================
# STRATEGY ENGINE
# ============================================================================

class StrategyEngine:
    """Generate trading signals"""
    
    def __init__(self, tickers, lookback_days=252):
        self.tickers = tickers
        self.lookback_days = lookback_days
    
    def calculate_signals(self):
        """Calculate value + momentum + quality scores"""
        
        print(f"\n{'='*70}")
        print(f"STRATEGY ENGINE - Analyzing {len(self.tickers)} stocks")
        print(f"{'='*70}")
        
        # Fetch data
        end_date = datetime.now()
        start_date = end_date - timedelta(days=self.lookback_days + 50)
        
        price_data = yf.download(self.tickers, start=start_date, end=end_date, progress=False)
        
        # Handle multi-index columns
        if isinstance(price_data.columns, pd.MultiIndex):
            if 'Adj Close' in price_data.columns.get_level_values(0):
                price_data = price_data['Adj Close']
            else:
                price_data = price_data['Close']
        elif 'Adj Close' in price_data.columns:
            price_data = price_data['Adj Close']
        else:
            price_data = price_data['Close']
        
        # Convert to DataFrame if Series
        if isinstance(price_data, pd.Series):
            price_data = price_data.to_frame(name=self.tickers[0])
        
        price_data = price_data.fillna(method='ffill').fillna(method='bfill')
        returns = price_data.pct_change()
        
        scores = {}
        
        for ticker in self.tickers:
            try:
                prices = price_data[ticker]
                rets = returns[ticker].dropna()
                
                # Momentum (40%)
                mom_6m = prices.pct_change(126).iloc[-1] if len(prices) > 126 else 0
                mom_12m = prices.pct_change(252).iloc[-1] if len(prices) > 252 else 0
                momentum = 0.6 * mom_6m + 0.4 * mom_12m
                
                # Quality (30%) - Sharpe ratio
                mean_ret = rets.mean() * 252
                vol = rets.std() * np.sqrt(252)
                quality = mean_ret / vol if vol > 0 else 0
                
                # Value (30%) - Inverse volatility
                value = 1 / vol if vol > 0 else 0
                
                # Composite score
                composite = 0.4 * momentum + 0.3 * quality + 0.3 * value
                
                scores[ticker] = composite
                
            except Exception as e:
                print(f"⚠️  Error analyzing {ticker}: {str(e)}")
                scores[ticker] = -999
        
        # Rank stocks
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        
        print(f"\nStock Rankings:")
        for i, (ticker, score) in enumerate(ranked[:8], 1):
            print(f"  {i}. {ticker}: {score:.4f}")
        
        # Select top 5 stocks
        top_tickers = [t for t, _ in ranked[:5]]
        
        # Calculate weights using inverse variance
        selected_returns = returns[top_tickers].iloc[-63:].dropna()  # Last 3 months
        variances = selected_returns.var()
        inv_var = 1 / variances
        raw_weights = inv_var / inv_var.sum()
        
        # Apply half-Kelly for safety
        weights = raw_weights * 0.5
        
        # Apply 10% position limit
        weights = np.clip(weights, 0, 0.10)
        weights = weights / weights.sum()
        
        target_allocation = dict(zip(top_tickers, weights))
        
        print(f"\nTarget Allocation:")
        for ticker, weight in sorted(target_allocation.items(), key=lambda x: x[1], reverse=True):
            print(f"  {ticker}: {weight:.2%}")
        
        return target_allocation


def run_live_paper_trading(api_key=None, secret_key=None):
    """Execute live paper trading with Alpaca"""
    
    print("\n" + "="*70)
    print("HORIZON LIVE PAPER TRADING - ALPACA API")
    print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*70)
    
    # Stock universe
    tickers = [
        'MSFT', 'AMZN', 'GOOG', 'CEG', 'NEE', 'OKLO',
        'ASML', 'AVGO', 'VRT', 'ETN', 'AWK'
    ]
    
    # Initialize Alpaca connection
    alpaca = AlpacaPaperTrading(api_key=api_key, secret_key=secret_key)
    
    # Get account status
    alpaca.get_account()
    
    # Run strategy
    strategy = StrategyEngine(tickers, lookback_days=252)
    target_allocation = strategy.calculate_signals()
    
    # Rebalance portfolio
    alpaca.rebalance_to_target(target_allocation)
    
    alpaca.get_account()
    
    print(f"\n{'='*70}")
    print("✓ Trading complete - Run Cell 6 to view performance")
    print(f"{'='*70}")
    
    return alpaca

# ============================================================================
# PERFORMANCE TRACKER
# ============================================================================

def get_performance_history(api_key=None, secret_key=None):
    """Get historical performance from Alpaca"""
    
    alpaca = AlpacaPaperTrading(api_key=api_key, secret_key=secret_key)
    
    # Get portfolio history (last 30 days)
    try:
        portfolio_history = alpaca.api.get_portfolio_history(
            period='1M',
            timeframe='1D'
        )
        
        print(f"\n{'='*70}")
        print("PERFORMANCE SUMMARY (Last 30 Days)")
        print(f"{'='*70}")
        
        equity = portfolio_history.equity
        
        if len(equity) > 1:
            initial_value = equity[0]
            final_value = equity[-1]
            total_return = (final_value - initial_value) / initial_value
            
            print(f"Initial Value: ${initial_value:,.2f}")
            print(f"Current Value: ${final_value:,.2f}")
            print(f"Total Return: {total_return:+.2%}")
            
            # Plot
            import matplotlib.pyplot as plt
            
            timestamps = portfolio_history.timestamp
            returns = [(e - equity[0]) / equity[0] * 100 for e in equity]
            
            plt.figure(figsize=(12, 6))
            plt.plot(timestamps, returns, linewidth=2, color='#2E86C1')
            plt.title('Alpaca Paper Trading Performance', fontsize=14, fontweight='bold')
            plt.xlabel('Date', fontsize=12)
            plt.ylabel('Return (%)', fontsize=12)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig('alpaca_performance.png', dpi=300)
            print("\n✓ Saved: alpaca_performance.png")
            plt.show()
        else:
            print("Not enough data yet. Trade for a few days first.")
    
    except Exception as e:
        print(f"Error fetching performance: {str(e)}")


print("✓ All functions loaded successfully")

✓ All functions loaded successfully


In [36]:
#API Keys
api_key = 'PKORGNKZIEVIEWQ7HO6TD4BPCI' 
secret_key = 'Bss2jxfYoXYrWMrvryLEhwt7EX9az8Q7NuLJ4qGgVTPa'
paper = True # Use paper trading environment

print("✓ API keys set successfully.")

✓ API keys set successfully.


In [37]:
alpaca = run_live_paper_trading(
    api_key= 'PKORGNKZIEVIEWQ7HO6TD4BPCI',
    secret_key='Bss2jxfYoXYrWMrvryLEhwt7EX9az8Q7NuLJ4qGgVTPa'
)


HORIZON LIVE PAPER TRADING - ALPACA API
Date: 2026-02-01 23:18:57
✓ Connected to Alpaca Paper Trading

ALPACA PAPER TRADING ACCOUNT
Buying Power: $200,000.00
Cash: $100,000.00
Portfolio Value: $100,000.00
Account Status: ACTIVE

STRATEGY ENGINE - Analyzing 11 stocks


/var/folders/np/35rvmxqn34z0tw1r9rdvqzbr0000gn/T/ipykernel_90284/2284824536.py:231: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  price_data = price_data.fillna(method='ffill').fillna(method='bfill')



Stock Rankings:
  1. GOOG: 2.2395
  2. ASML: 1.8115
  3. NEE: 1.7586
  4. VRT: 1.4331
  5. MSFT: 1.4175
  6. ETN: 1.3561
  7. AVGO: 1.3246
  8. AMZN: 1.2829

Target Allocation:
  GOOG: 27.76%
  NEE: 27.76%
  MSFT: 25.24%
  ASML: 12.23%
  VRT: 7.00%

REBALANCING TO TARGET ALLOCATION

Total Portfolio Value: $100,000.00

No current positions

Target Allocation:
  GOOG: 27.76% ($27,760.90)
  NEE: 27.76% ($27,760.90)
  MSFT: 25.24% ($25,244.61)
  ASML: 12.23% ($12,233.24)
  VRT: 7.00% ($7,000.35)

Executing Trades:
✓ buy order placed: 82 shares of GOOG
✓ buy order placed: 8 shares of ASML
✓ buy order placed: 315 shares of NEE
✓ buy order placed: 37 shares of VRT
✓ buy order placed: 58 shares of MSFT

✓ Rebalancing complete

No current positions

ALPACA PAPER TRADING ACCOUNT
Buying Power: $101,500.85
Cash: $100,000.00
Portfolio Value: $100,000.00
Account Status: ACTIVE

✓ Trading complete - Run Cell 6 to view performance


In [44]:
get_performance_history(
    api_key= 'PKORGNKZIEVIEWQ7HO6TD4BPCI',
    secret_key= 'Bss2jxfYoXYrWMrvryLEhwt7EX9az8Q7NuLJ4qGgVTPa'
)

✓ Connected to Alpaca Paper Trading

PERFORMANCE SUMMARY (Last 30 Days)
Error fetching performance: float division by zero


In [41]:
# Risk Adjusted Performance Metrics

def calculate_performance_metrics(api_key, secret_key, risk_free_rate=0.045):
    """Calculate Sharpe, Sortino, Calmar, and other metrics"""
    
    alpaca = AlpacaPaperTrading(api_key=api_key, secret_key=secret_key)
    
    try:
        portfolio_history = alpaca.api.get_portfolio_history(period='1M', timeframe='1D')
        equity = np.array(portfolio_history.equity)
        
        if len(equity) < 2:
            print("Need at least 2 days of data")
            return None
        
        returns = np.diff(equity) / equity[:-1]
        trading_days = 252
        
        print(f"\n{'='*70}")
        print("RISK-ADJUSTED PERFORMANCE METRICS")
        print(f"{'='*70}")
        
        total_return = (equity[-1] - equity[0]) / equity[0]
        avg_daily_return = np.mean(returns)
        daily_vol = np.std(returns)
        annual_return = avg_daily_return * trading_days
        annual_vol = daily_vol * np.sqrt(trading_days)
        
        print(f"\nBasic Statistics:")
        print(f"  Days Tracked: {len(equity)}")
        print(f"  Total Return: {total_return:.2%}")
        print(f"  Annual Return: {annual_return:.2%}")
        print(f"  Annual Volatility: {annual_vol:.2%}")
        
        sharpe = (annual_return - risk_free_rate) / annual_vol if annual_vol > 0 else 0
        print(f"\nSharpe Ratio: {sharpe:.3f}")
        print(f"  (> 1.0 = Good, > 2.0 = Very Good, > 3.0 = Excellent)")
        
        downside_returns = returns[returns < 0]
        downside_vol = np.std(downside_returns) * np.sqrt(trading_days) if len(downside_returns) > 0 else 0
        sortino = (annual_return - risk_free_rate) / downside_vol if downside_vol > 0 else 0
        print(f"\nSortino Ratio: {sortino:.3f}")
        
        cumulative = np.cumprod(1 + returns)
        running_max = np.maximum.accumulate(cumulative)
        drawdown = (cumulative - running_max) / running_max
        max_drawdown = np.min(drawdown)
        print(f"\nMax Drawdown: {max_drawdown:.2%}")
        
        calmar = annual_return / abs(max_drawdown) if max_drawdown != 0 else 0
        print(f"Calmar Ratio: {calmar:.3f}")
        
        win_rate = np.sum(returns > 0) / len(returns)
        print(f"\nWin Rate: {win_rate:.2%}")
        
        print(f"\n{'='*70}")
        
        return {
            'sharpe': sharpe, 'sortino': sortino, 'max_drawdown': max_drawdown,
            'calmar': calmar, 'annual_return': annual_return, 'annual_vol': annual_vol
        }
    except Exception as e:
        print(f"Error: {str(e)}")
        return None

In [43]:
# 5 Year Backtest Performance Eval with Statistical Significance

def run_5year_backtest_with_stats(tickers, initial_capital=100000):
    """
    Run 5-year backtest with statistical significance testing
    Compare to SPY benchmark
    """
    from scipy import stats
    
    print(f"\n{'='*70}")
    print("5-YEAR BACKTEST WITH STATISTICAL SIGNIFICANCE")
    print(f"{'='*70}")
    print(f"Universe: {len(tickers)} stocks")
    print(f"Initial Capital: ${initial_capital:,.0f}")
    
    # Fetch 5 years of data
    end_date = datetime.now()
    start_date = end_date - timedelta(days=5*365 + 100)
    
    print(f"\nFetching data from {start_date.date()} to {end_date.date()}...")
    
    # Get stock data
    stock_data = yf.download(tickers, start=start_date, end=end_date, progress=False)
    
    if isinstance(stock_data.columns, pd.MultiIndex):
        stock_data = stock_data['Adj Close'] if 'Adj Close' in stock_data.columns.get_level_values(0) else stock_data['Close']
    
    if isinstance(stock_data, pd.Series):
        stock_data = stock_data.to_frame()
    
    stock_data = stock_data.fillna(method='ffill').fillna(method='bfill')
    
    # Get SPY benchmark
    spy_data = yf.download('SPY', start=start_date, end=end_date, progress=False)['Adj Close']
    
    print(f"✓ Downloaded {len(stock_data)} days of data")
    
    # Walk-forward backtest
    train_window = 252  # 1 year
    test_window = 63    # 3 months
    
    portfolio_values = [initial_capital]
    spy_values = [initial_capital]
    dates = []
    
    start_idx = train_window
    
    while start_idx + test_window <= len(stock_data):
        # Training period
        train_data = stock_data.iloc[start_idx - train_window:start_idx]
        train_returns = train_data.pct_change().dropna()
        
        # Score stocks
        scores = {}
        for ticker in train_data.columns:
            try:
                prices = train_data[ticker]
                rets = train_returns[ticker]
                
                mom = 0.6 * prices.pct_change(63).iloc[-1] + 0.4 * prices.pct_change(126).iloc[-1]
                sharpe = (rets.mean() * 252) / (rets.std() * np.sqrt(252))
                value = 1 / (rets.std() * np.sqrt(252))
                
                scores[ticker] = 0.4 * mom + 0.3 * sharpe + 0.3 * value
            except:
                scores[ticker] = -999
        
        # Select top 5
        top_tickers = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]
        top_tickers = [t for t, _ in top_tickers]
        
        # Calculate weights
        selected_returns = train_returns[top_tickers].iloc[-63:]
        variances = selected_returns.var()
        weights = (1 / variances) / (1 / variances).sum()
        weights = np.clip(weights * 0.5, 0, 0.10)
        weights = weights / weights.sum()
        
        # Test period
        test_data = stock_data[top_tickers].iloc[start_idx:start_idx + test_window]
        test_returns = test_data.pct_change().dropna()
        
        # Portfolio return
        port_return = (test_returns @ weights).sum()
        portfolio_values.append(portfolio_values[-1] * (1 + port_return))
        
        # SPY return
        spy_return = spy_data.iloc[start_idx:start_idx + test_window].pct_change().sum()
        spy_values.append(spy_values[-1] * (1 + spy_return))
        
        dates.append(stock_data.index[start_idx + test_window])
        
        start_idx += test_window
    
    # Convert to returns
    portfolio_returns = np.diff(portfolio_values) / portfolio_values[:-1]
    spy_returns = np.diff(spy_values) / spy_values[:-1]
    
    # Calculate metrics
    print(f"\n{'='*70}")
    print("BACKTEST RESULTS")
    print(f"{'='*70}")
    
    port_total = (portfolio_values[-1] - portfolio_values[0]) / portfolio_values[0]
    spy_total = (spy_values[-1] - spy_values[0]) / spy_values[0]
    
    port_cagr = (portfolio_values[-1] / portfolio_values[0]) ** (1/5) - 1
    spy_cagr = (spy_values[-1] / spy_values[0]) ** (1/5) - 1
    
    port_vol = np.std(portfolio_returns) * np.sqrt(252/test_window)
    spy_vol = np.std(spy_returns) * np.sqrt(252/test_window)
    
    port_sharpe = (port_cagr - 0.045) / port_vol
    spy_sharpe = (spy_cagr - 0.045) / spy_vol
    
    print(f"\nStrategy Performance:")
    print(f"  Total Return: {port_total:.2%}")
    print(f"  CAGR: {port_cagr:.2%}")
    print(f"  Volatility: {port_vol:.2%}")
    print(f"  Sharpe Ratio: {port_sharpe:.3f}")
    
    print(f"\nSPY Benchmark:")
    print(f"  Total Return: {spy_total:.2%}")
    print(f"  CAGR: {spy_cagr:.2%}")
    print(f"  Volatility: {spy_vol:.2%}")
    print(f"  Sharpe Ratio: {spy_sharpe:.3f}")
    
    print(f"\nOutperformance:")
    print(f"  Alpha: {port_cagr - spy_cagr:+.2%}")
    print(f"  Sharpe Diff: {port_sharpe - spy_sharpe:+.3f}")
    
    # Statistical Significance Testing
    print(f"\n{'='*70}")
    print("STATISTICAL SIGNIFICANCE TESTS")
    print(f"{'='*70}")
    
    # 1. T-test: Are returns significantly different from SPY?
    t_stat, p_value = stats.ttest_rel(portfolio_returns, spy_returns)
    print(f"\nPaired T-Test (Strategy vs SPY):")
    print(f"  t-statistic: {t_stat:.3f}")
    print(f"  p-value: {p_value:.4f}")
    if p_value < 0.05:
        print(f"  ✓ SIGNIFICANT at 95% confidence level")
        print(f"    Strategy returns are statistically different from SPY")
    else:
        print(f"  ✗ NOT SIGNIFICANT")
        print(f"    Cannot reject that returns = SPY")
    
    # 2. Sharpe Ratio Test
    n = len(portfolio_returns)
    sharpe_diff = port_sharpe - spy_sharpe
    se_diff = np.sqrt((1 + 0.5 * port_sharpe**2) / n + (1 + 0.5 * spy_sharpe**2) / n)
    z_score = sharpe_diff / se_diff
    p_sharpe = 2 * (1 - stats.norm.cdf(abs(z_score)))
    
    print(f"\nSharpe Ratio Difference Test:")
    print(f"  Sharpe Difference: {sharpe_diff:+.3f}")
    print(f"  z-score: {z_score:.3f}")
    print(f"  p-value: {p_sharpe:.4f}")
    if p_sharpe < 0.05:
        print(f"  ✓ SIGNIFICANT at 95% confidence level")
        print(f"    Sharpe ratio is statistically superior to SPY")
    else:
        print(f"  ✗ NOT SIGNIFICANT")
    
    # 3. Information Ratio
    excess_returns = portfolio_returns - spy_returns
    info_ratio = np.mean(excess_returns) / np.std(excess_returns) * np.sqrt(252/test_window)
    
    print(f"\nInformation Ratio:")
    print(f"  IR: {info_ratio:.3f}")
    print(f"  (Measures consistency of outperformance)")
    
    # 4. Win Rate
    win_rate = np.sum(portfolio_returns > spy_returns) / len(portfolio_returns)
    print(f"\nWin Rate vs SPY: {win_rate:.2%}")
    
    # Confidence Intervals
    port_ci = stats.t.interval(0.95, len(portfolio_returns)-1,
                               loc=np.mean(portfolio_returns),
                               scale=stats.sem(portfolio_returns))
    
    print(f"\n95% Confidence Interval (Period Returns):")
    print(f"  Strategy: [{port_ci[0]:.2%}, {port_ci[1]:.2%}]")
    
    # Plot results
    plt.figure(figsize=(14, 8))
    
    plt.subplot(2, 1, 1)
    plt.plot(dates, portfolio_values[1:], label='Strategy', linewidth=2)
    plt.plot(dates, spy_values[1:], label='SPY Benchmark', linewidth=2, alpha=0.7)
    plt.title('5-Year Backtest: Strategy vs SPY', fontsize=14, fontweight='bold')
    plt.ylabel('Portfolio Value ($)', fontsize=12)
    plt.legend(loc='upper left')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 1, 2)
    cumulative_alpha = np.array(portfolio_values[1:]) - np.array(spy_values[1:])
    plt.plot(dates, cumulative_alpha, color='green', linewidth=2)
    plt.axhline(0, color='black', linestyle='--', alpha=0.5)
    plt.title('Cumulative Alpha (Strategy - SPY)', fontsize=14, fontweight='bold')
    plt.ylabel('Alpha ($)', fontsize=12)
    plt.xlabel('Date', fontsize=12)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('5year_backtest.png', dpi=300)
    print(f"\n✓ Saved: 5year_backtest.png")
    plt.show()
    
    return {
        'strategy_cagr': port_cagr,
        'spy_cagr': spy_cagr,
        'alpha': port_cagr - spy_cagr,
        'strategy_sharpe': port_sharpe,
        'spy_sharpe': spy_sharpe,
        'p_value_returns': p_value,
        'p_value_sharpe': p_sharpe,
        'info_ratio': info_ratio,
        'win_rate': win_rate
    }